# 05 Estimate the founding model

This notebook estimates a negative-binomial model for quarterly firm births in 100 m grid cells. It uses the outputs produced by `100m_data_foundation.ipynb` and the routing workflow.

Run it only after routing notebooks 04 and 05 have finished for every year from 2015 to 2025. The preflight below stops before loading data if a yearly product is missing or has the wrong schema.

The stored accessibility measures are cumulative. The notebook removes the origin-cell mass and converts the contours into non-overlapping rings. Legitimately skipped routing origins are excluded and reported.

## Configuration

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "TOOLS").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or one of its subdirectories.")


PROJECT_DIR = discover_project_dir()
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
FEATURE_ROOT = ANAL_DATA / "routing" / "features"
PANEL_PATH = ANAL_DATA / "raster_quarter_panel_100m.parquet"
MODEL_DIR = ANAL_DATA / "models"

START_YEAR, END_YEAR = 2015, 2025
YEARS = list(range(START_YEAR, END_YEAR + 1))
CAR_CONTOURS = [5, 10, 15, 30]
WALK_CONTOURS = [5, 10]
SUBTRACT_OWN_CELL = True

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

## 1 Preflight

In [2]:
panel_required = {"grid_id", "municipality_id", "year", "quarter", "period", "births", "active_firms_tminus1"}
main_required = {"grid_id", "year", "quarter", "period", "own_cell_pop", "own_cell_firms"}
main_required |= {f"pop_access_{m}min" for m in CAR_CONTOURS}
main_required |= {f"existing_firms_access_{m}min" for m in CAR_CONTOURS}
main_required |= {f"reachable_cells_{m}min" for m in CAR_CONTOURS}
walk_required = {"grid_id", "year", "quarter", "period", "own_cell_walk_pop", "own_cell_walk_firms", "pt_ohne_haltestelle"}
walk_required |= {column for m in WALK_CONTOURS for column in [f"walk_pop_{m}min", f"walk_firms_{m}min", f"walk_pt_routes_{m}min", f"reachable_cells_walk_{m}min"]}
nearest_required = {"grid_id", "year", "tt_rail_station_min", "tt_motorway_exit_min"}

required_files = {PANEL_PATH: panel_required}
for year in YEARS:
    year_dir = FEATURE_ROOT / str(year)
    required_files[year_dir / "accessibility_potentials_100m.parquet"] = main_required
    required_files[year_dir / "pedestrian_accessibility_quarter_100m.parquet"] = walk_required
    required_files[year_dir / "nearest_infrastructure_100m.parquet"] = nearest_required

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Routing is not complete. Missing files: {missing_files}")

for path, required_columns in required_files.items():
    columns = set(pq.ParquetFile(path).schema_arrow.names)
    missing_columns = sorted(required_columns - columns)
    if missing_columns:
        raise ValueError(f"{path.name} is missing columns: {missing_columns}")

print(f"Preflight passed for {len(YEARS)} routing years.")

Preflight passed for 11 routing years.


## 2 Load the model table

In [3]:
def feature_glob(filename: str) -> str:
    return (FEATURE_ROOT / "*" / filename).as_posix()


car_columns = ", ".join(
    column
    for m in CAR_CONTOURS
    for column in [f"a.pop_access_{m}min", f"a.existing_firms_access_{m}min", f"a.reachable_cells_{m}min"]
)
walk_columns = ", ".join(
    column
    for m in WALK_CONTOURS
    for column in [f"w.walk_pop_{m}min", f"w.walk_firms_{m}min", f"w.walk_pt_routes_{m}min", f"w.reachable_cells_walk_{m}min"]
)

query = f"""
SELECT
    p.grid_id, p.municipality_id, p.year, p.quarter, p.period,
    p.births, p.active_firms_tminus1,
    a.own_cell_pop, a.own_cell_firms,
    w.own_cell_walk_pop, w.own_cell_walk_firms,
    {car_columns},
    {walk_columns},
    w.pt_ohne_haltestelle,
    n.tt_rail_station_min, n.tt_motorway_exit_min
FROM read_parquet('{PANEL_PATH.as_posix()}') p
JOIN read_parquet('{feature_glob("accessibility_potentials_100m.parquet")}') a
  USING (grid_id, year, quarter, period)
JOIN read_parquet('{feature_glob("pedestrian_accessibility_quarter_100m.parquet")}') w
  USING (grid_id, year, quarter, period)
JOIN read_parquet('{feature_glob("nearest_infrastructure_100m.parquet")}') n
  USING (grid_id, year)
WHERE p.year BETWEEN {START_YEAR} AND {END_YEAR}
ORDER BY p.grid_id, p.year, p.quarter
"""

expected_rows = con.execute(
    f"SELECT count(*) FROM read_parquet('{PANEL_PATH.as_posix()}') WHERE year BETWEEN {START_YEAR} AND {END_YEAR}"
).fetchone()[0]
df = con.execute(query).df()

assert len(df) == expected_rows, "The routing join lost or duplicated panel rows."
assert not df.duplicated(["grid_id", "year", "quarter"]).any()
print(f"{len(df):,} rows, {df['grid_id'].nunique():,} cells, {df['period'].nunique()} quarters")

5,520,812 rows, 125,473 cells, 44 quarters


### Remove incomplete routed observations

In [4]:
routing_columns = [
    *[f"pop_access_{m}min" for m in CAR_CONTOURS],
    *[f"existing_firms_access_{m}min" for m in CAR_CONTOURS],
    *[f"walk_pop_{m}min" for m in WALK_CONTOURS],
    *[f"walk_firms_{m}min" for m in WALK_CONTOURS],
    "walk_pt_routes_10min", "tt_rail_station_min", "tt_motorway_exit_min",
]
incomplete = df[routing_columns].isna().any(axis=1)
print(f"Excluded incomplete routed observations: {incomplete.sum():,} ({incomplete.mean():.3%})")
df = df.loc[~incomplete].copy()

Excluded incomplete routed observations: 316 (0.006%)


## 3 Build non-overlapping accessibility rings

In [5]:
def build_rings(frame: pd.DataFrame, prefix: str, contours: list[int], own_column: str) -> pd.DataFrame:
    cumulative = {m: frame[f"{prefix}_{m}min"].to_numpy(dtype="float64") for m in contours}
    if SUBTRACT_OWN_CELL:
        own = frame[own_column].to_numpy(dtype="float64")
        cumulative = {m: np.clip(values - own, 0, None) for m, values in cumulative.items()}
    rings = {}
    lower = 0
    for m in contours:
        previous = 0.0 if lower == 0 else cumulative[lower]
        rings[f"{prefix}_ring_{lower}_{m}"] = np.clip(cumulative[m] - previous, 0, None)
        lower = m
    return pd.DataFrame(rings, index=frame.index)


rings = pd.concat(
    [
        build_rings(df, "pop_access", CAR_CONTOURS, "own_cell_pop"),
        build_rings(df, "existing_firms_access", CAR_CONTOURS, "own_cell_firms"),
        build_rings(df, "walk_pop", WALK_CONTOURS, "own_cell_walk_pop"),
        build_rings(df, "walk_firms", WALK_CONTOURS, "own_cell_walk_firms"),
    ],
    axis=1,
)
df = pd.concat([df, rings], axis=1)
ring_columns = rings.columns.tolist()
for column in ring_columns:
    df[f"log_{column}"] = np.log1p(df[column])

print(ring_columns)

['pop_access_ring_0_5', 'pop_access_ring_5_10', 'pop_access_ring_10_15', 'pop_access_ring_15_30', 'existing_firms_access_ring_0_5', 'existing_firms_access_ring_5_10', 'existing_firms_access_ring_10_15', 'existing_firms_access_ring_15_30', 'walk_pop_ring_0_5', 'walk_pop_ring_5_10', 'walk_firms_ring_0_5', 'walk_firms_ring_5_10']


## 4 Build the design matrix

In [6]:
df["log_own_firms"] = np.log1p(df["active_firms_tminus1"])
df["log_own_pop"] = np.log1p(df["own_cell_pop"])
df["log_tt_rail_station"] = np.log1p(df["tt_rail_station_min"])
df["log_tt_motorway_exit"] = np.log1p(df["tt_motorway_exit_min"])

df = df.sort_values(["grid_id", "year", "quarter"])
period_number = df["year"] * 4 + df["quarter"]
previous_period_number = period_number.groupby(df["grid_id"], sort=False).shift(4)
previous_population = df.groupby("grid_id", sort=False)["own_cell_pop"].shift(4)
previous_population = previous_population.where(period_number - previous_period_number == 4)
df["population_growth_yoy"] = (df["own_cell_pop"] - previous_population) / previous_population
df["population_growth_yoy"] = df["population_growth_yoy"].replace([np.inf, -np.inf], np.nan)

regressors = [
    "log_own_firms", "log_own_pop",
    *[f"log_{column}" for column in ring_columns],
    "population_growth_yoy", "log_tt_rail_station", "log_tt_motorway_exit",
    "walk_pt_routes_10min", "pt_ohne_haltestelle",
]

before = len(df)
df = df.dropna(subset=regressors).copy()
print(f"Excluded for unavailable year-on-year growth or covariates: {before - len(df):,}")

period_dummies = pd.get_dummies(df["period"], prefix="period", drop_first=True, dtype="float64")
X = pd.concat([df[regressors].astype("float64"), period_dummies], axis=1)
X.insert(0, "const", 1.0)
y = df["births"].astype("float64")
clusters = df["grid_id"]

assert np.isfinite(X.to_numpy()).all()
assert (y >= 0).all()
print(f"Design matrix: {X.shape[0]:,} x {X.shape[1]}; births: {y.sum():,.0f}; zero share: {(y == 0).mean():.1%}")

Excluded for unavailable year-on-year growth or covariates: 590,856
Design matrix: 4,929,640 x 59; births: 613; zero share: 100.0%


### Simple collinearity check

In [7]:
correlations = X[regressors].corr()
upper = correlations.where(np.triu(np.ones(correlations.shape), k=1).astype(bool)).stack()
high_correlations = upper[upper.abs() > 0.9].sort_values(key=abs, ascending=False)
print(high_correlations.to_string() if len(high_correlations) else "No absolute correlation above 0.9.")

log_pop_access_ring_15_30  log_existing_firms_access_ring_15_30    0.974472
log_pop_access_ring_10_15  log_existing_firms_access_ring_10_15    0.954806
log_pop_access_ring_5_10   log_existing_firms_access_ring_5_10     0.938631


## 5 Estimate NB2

The negative-binomial model uses cell-clustered standard errors. A Poisson model provides the boundary likelihood-ratio test for overdispersion.

In [ ]:
from scipy import stats
from statsmodels.discrete.discrete_model import NegativeBinomial, Poisson

poisson_result = Poisson(y, X).fit(maxiter=200, disp=0)
nb_result = NegativeBinomial(y, X, loglike_method="nb2").fit(
    start_params=np.append(poisson_result.params.to_numpy(), 0.1),
    cov_type="cluster",
    cov_kwds={"groups": clusters},
    maxiter=500,
    disp=1,
)

lr_statistic = max(0.0, 2 * (nb_result.llf - poisson_result.llf))
lr_p_value = 0.5 * stats.chi2.sf(lr_statistic, df=1)
print(f"Boundary LR test for alpha = 0: statistic={lr_statistic:.2f}, p={lr_p_value:.3g}")
print(nb_result.summary())

d:\CO2_Masterarbeit\CO2_Masterarbeit\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Optimization terminated successfully.
         Current function value: 0.001204
         Iterations: 0
         Function evaluations: 1
         Gradient evaluations: 1


d:\CO2_Masterarbeit\CO2_Masterarbeit\.venv\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


Boundary LR test for alpha = 0: statistic=0.00, p=0.5


d:\CO2_Masterarbeit\CO2_Masterarbeit\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3739: RuntimeWarning: invalid value encountered in log
  start_params[-1] = np.log(start_params[-1])


## 6 Save results

In [ ]:
confidence = nb_result.conf_int()
results = pd.DataFrame(
    {
        "coefficient": nb_result.params,
        "std_error": nb_result.bse,
        "p_value": nb_result.pvalues,
        "ci_lower": confidence.iloc[:, 0],
        "ci_upper": confidence.iloc[:, 1],
    }
)
results["irr"] = np.exp(results["coefficient"])
results["irr_ci_lower"] = np.exp(results["ci_lower"])
results["irr_ci_upper"] = np.exp(results["ci_upper"])
alpha = float(results.loc["alpha", "coefficient"])
results.loc["alpha", ["irr", "irr_ci_lower", "irr_ci_upper"]] = np.nan
reported = results.loc[~results.index.str.startswith("period_")].copy()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
reported.to_csv(MODEL_DIR / "founding_nb2_results.csv", index_label="term")
pd.Series(
    {"n_observations": len(df), "n_cells": df['grid_id'].nunique(), "alpha": alpha, "lr_statistic": lr_statistic, "lr_p_value": lr_p_value}
).to_csv(MODEL_DIR / "founding_nb2_diagnostics.csv", header=["value"])
reported.round(4)